# Vector Database using FAISS

## Introduction

A **Vector Database** is a specialized database designed to store and retrieve high-dimensional vectors called **embeddings**. These embeddings represent the semantic meaning of text, images, audio, or any unstructured data.

Unlike traditional databases that perform exact keyword matching, vector databases perform **semantic similarity search**, allowing the system to retrieve information based on meaning rather than exact words.

### Why Do We Need a Vector Database?

- Stores high-dimensional embeddings efficiently.
- Performs fast similarity search on millions of vectors.
- Retrieves semantically similar documents.
- Forms the retrieval layer in Retrieval-Augmented Generation (RAG).

## Step 1: Install Required Library

FAISS (Facebook AI Similarity Search) is an open-source library developed by Meta for storing and searching high-dimensional vectors efficiently. It is optimized for similarity search and is widely used in Retrieval-Augmented Generation (RAG), semantic search, recommendation systems, and document retrieval.

### Purpose

Install the FAISS library to create and manage the vector database.

In [3]:
# pip install faiss-cpu

## Step 2: Import Required Libraries

The notebook requires multiple libraries to perform different tasks during the vector database pipeline.

- **spaCy** is used for Natural Language Processing (NLP).
- **FAISS** creates and searches the vector database.
- **RecursiveCharacterTextSplitter** divides large documents into smaller overlapping chunks.
- **SentenceTransformer** converts text into dense vector embeddings.

### Purpose

Import all the required libraries for preprocessing, embedding generation, and vector indexing.

In [4]:
import spacy
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

c:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 3: Load the spaCy Language Model

The English language model contains pretrained components such as tokenization, part-of-speech tagging, lemmatization, and named entity recognition.

Although this notebook mainly focuses on vector databases, loading the language model prepares the environment for any NLP preprocessing that may be required later.

### Purpose

Load the pretrained English language model.

In [5]:
nlp = spacy.load("en_core_web_sm")

## Step 4: Read the Dataset

The first step in building a vector database is to load the dataset into memory. The dataset can be a text file, PDF, CSV, or any other document containing information that needs to be indexed for semantic search.

In this notebook, the dataset consists of machine learning sentences stored in a text file. These sentences will later be divided into smaller chunks and converted into embeddings.

### Purpose

Read the text file and store its contents as a single string.

In [6]:
data = open("machine_learning_2000_sentences.txt").read()

## Step 5: Split the Document into Chunks

Large documents cannot be directly converted into embeddings because embedding models have a maximum input size. Therefore, the document is divided into smaller pieces called **chunks**.

Chunking improves retrieval quality because the vector database searches only the relevant portions of a document instead of the entire document.

In this notebook, **RecursiveCharacterTextSplitter** from LangChain is used. It recursively splits the document while preserving as much context as possible.

### Advantages of Chunking

- Reduces memory usage.
- Improves semantic search accuracy.
- Preserves contextual information.
- Makes retrieval faster.
- Prevents embedding model input limits from being exceeded.

### Purpose

Create a text splitter for dividing the document into overlapping chunks.

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

## Understanding Chunk Size and Chunk Overlap

### Chunk Size

`chunk_size` specifies the maximum number of characters allowed in a single chunk.

Example:

If `chunk_size = 100`, then each chunk contains approximately 100 characters.

A smaller chunk provides more precise retrieval, whereas a larger chunk preserves more context.

---

### Chunk Overlap

`chunk_overlap` specifies the number of characters shared between two consecutive chunks.

The overlap helps preserve context between neighboring chunks.

For example,

Chunk 1

```
Machine Learning is a subset of Artificial Intelligence.
```

Chunk 2 (20-character overlap)

```
Artificial Intelligence focuses on building intelligent systems.
```

Notice that part of the previous chunk is repeated in the next chunk.

### Why Do We Use Overlap?

Without overlap, important information located at the boundary between two chunks may be lost.

Using overlap helps:

- Preserve context.
- Improve retrieval accuracy.
- Reduce information loss.
- Produce better RAG responses.

## Step 6: Generate Text Chunks

After creating the text splitter, the complete document is divided into multiple smaller chunks.

Each chunk will later be converted into an embedding and stored inside the vector database.

### Purpose

Split the document into overlapping text chunks.

In [8]:
chunks = splitter.split_text(data)

## Step 7: Display the Generated Chunks

Displaying the generated chunks helps verify whether the document has been split correctly.

It also allows us to inspect the chunk size, overlap, and formatting before generating embeddings.

### Purpose

Display the generated chunks and check their data types.

In [9]:
# print(chunks)
print(type(chunks))
print(type(chunks[0]))

<class 'list'>
<class 'str'>


# Embedding Generation

After chunking the document, the next step is to convert every chunk into a numerical representation called an **embedding**.

An embedding captures the semantic meaning of text, allowing the vector database to compare documents based on meaning rather than exact keywords.

Embedding models generate dense vectors where semantically similar sentences are located close to each other in the vector space.

## What is an Embedding?

An **embedding** is a dense numerical vector that represents the semantic meaning of text.

Unlike one-hot encoding or Bag of Words, embeddings capture relationships between words and sentences.

For example,

Sentence 1

```
Machine Learning is a subset of AI.
```

Sentence 2

```
Artificial Intelligence includes Machine Learning.
```

Although the wording is different, both sentences have similar meanings.

Their embeddings will therefore be located close together in the vector space.

### Advantages of Embeddings

- Capture semantic meaning.
- Support similarity search.
- Reduce dimensionality.
- Improve document retrieval.
- Used extensively in RAG systems.

## Why Do We Need Embeddings?

Computers cannot understand text directly.

Therefore, every text chunk is converted into a numerical vector before storing it inside the vector database.

During retrieval,

1. The user query is converted into an embedding.
2. The vector database compares it with stored embeddings.
3. The most similar embeddings are retrieved.
4. Their corresponding text chunks are returned.

Without embeddings, semantic similarity search is not possible.

## Step 8: Load the Sentence Transformer Model

Sentence Transformers are pretrained deep learning models that convert sentences into dense embeddings.

The model used in this notebook is **all-MiniLM-L6-v2**, which is lightweight, fast, and widely used for semantic search and Retrieval-Augmented Generation (RAG).

### Purpose

Load the pretrained embedding model.

In [10]:
embedding_model = SentenceTransformer(
    model_name_or_path="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:03<00:00, 27.45it/s]


## Step 9: Convert Chunks into Embeddings

Each chunk is passed through the embedding model.

The output is a dense vector representing the semantic meaning of that chunk.

The embeddings are then converted into **float32** because FAISS requires vectors in this format.

### Purpose

Generate vector embeddings for every text chunk.

In [11]:
embeddings = embedding_model.encode(chunks).astype("float32")

## Step 10: Display the Generated Embeddings

The generated embeddings are numerical vectors representing each text chunk.

Each row corresponds to one chunk, while each column represents one feature of the embedding space.

### Purpose

Display the generated embeddings.

In [12]:
embeddings

array([[-0.07728466, -0.00678861,  0.05914882, ...,  0.06669836,
        -0.07794894,  0.01822445],
       [ 0.01255796, -0.04760394,  0.02898123, ...,  0.04792494,
        -0.02147599, -0.02757207],
       [ 0.0154788 , -0.00400942, -0.09803507, ...,  0.04105116,
         0.03302541, -0.01764336],
       ...,
       [-0.01283604, -0.00381554,  0.05603732, ...,  0.08069879,
         0.06248713, -0.05116992],
       [-0.03425974, -0.02970343, -0.01779654, ...,  0.0584114 ,
         0.03584176, -0.00811812],
       [ 0.02394698,  0.00384395,  0.05797815, ..., -0.13486075,
         0.00705718,  0.03490416]], shape=(7240, 384), dtype=float32)

## Step 11: Find the Embedding Dimension

Every embedding model generates vectors of a fixed size.

This size is called the **embedding dimension**.

FAISS requires this dimension while creating the vector index because every vector stored inside the index must have the same number of features.

### Purpose

Determine the dimension of the embedding vectors.

In [13]:
dimension = embeddings.shape[1]
dimension

384

# Vector Database Creation using FAISS

After generating embeddings, the next step is to create a vector database.

A vector database stores embeddings and enables efficient similarity search. Instead of searching using exact keywords, it compares vectors based on their mathematical distance or similarity.

FAISS (Facebook AI Similarity Search) is one of the most popular vector databases used for semantic search, Retrieval-Augmented Generation (RAG), recommendation systems, and document retrieval.

The complete indexing process consists of the following steps:

1. Normalize the embeddings.
2. Create a FAISS index.
3. Store embeddings inside the index.
4. Convert the user query into an embedding.
5. Search for the most similar vectors.

# Vector Database Creation using FAISS

After generating embeddings, the next step is to create a vector database.

A vector database stores embeddings and enables efficient similarity search. Instead of searching using exact keywords, it compares vectors based on their mathematical distance or similarity.

FAISS (Facebook AI Similarity Search) is one of the most popular vector databases used for semantic search, Retrieval-Augmented Generation (RAG), recommendation systems, and document retrieval.

The complete indexing process consists of the following steps:

1. Normalize the embeddings.
2. Create a FAISS index.
3. Store embeddings inside the index.
4. Convert the user query into an embedding.
5. Search for the most similar vectors.

In [14]:
faiss.normalize_L2(embeddings)

## What is L2 Normalization?

L2 Normalization divides every element of a vector by its Euclidean length.

After normalization,

- Every vector has magnitude equal to **1**.
- Only the direction of the vector is preserved.
- Similarity depends only on semantic meaning.

For example,

Original Vector

```
[3, 4]
```

Magnitude

```
√(3² + 4²) = 5
```

Normalized Vector

```
[3/5, 4/5]

=

[0.6, 0.8]
```

Now the magnitude becomes

```
√(0.6² + 0.8²)

= 1
```

This allows cosine similarity to compare vectors fairly.

## Why is Normalization Required?

Suppose two vectors represent similar sentences.

Vector A

```
[100, 200]
```

Vector B

```
[1, 2]
```

Although both vectors point in the same direction, their magnitudes are very different.

Without normalization, the larger vector may appear more important simply because of its size.

Normalization removes this effect by making every vector have the same magnitude.

As a result, similarity depends only on the direction of the vectors, which represents their semantic meaning.

## Cosine Similarity vs Euclidean Distance

FAISS supports multiple similarity measures.

The two most common are **Cosine Similarity** and **Euclidean Distance (L2 Distance)**.

| Cosine Similarity | Euclidean Distance |
|-------------------|--------------------|
| Measures angle between vectors | Measures straight-line distance |
| Ignores vector magnitude | Uses vector magnitude |
| Higher score indicates greater similarity | Smaller distance indicates greater similarity |
| Requires normalization | Normalization is optional |
| Used for semantic search | Used for nearest-neighbor search |

In Retrieval-Augmented Generation (RAG), **Cosine Similarity** is the most commonly used similarity measure because it compares semantic meaning rather than vector length.

## Step 13: Create the FAISS Index

A FAISS Index is the data structure that stores all embedding vectors.

Once the embeddings are stored, FAISS can efficiently retrieve the vectors that are most similar to a given query.

The type of index determines how similarity is calculated.

This notebook uses **IndexFlatIP**, which performs **Inner Product** search. When vectors are normalized, Inner Product produces the same ranking as Cosine Similarity.

### Purpose

Create a FAISS index for storing embeddings.

In [15]:
index = faiss.IndexFlatIP(dimension)
# index = faiss.IndexFlatL2(dimension)

## Understanding IndexFlatIP

`IndexFlatIP` stands for **Index Flat Inner Product**.

It compares vectors using the Inner Product.

When embeddings are normalized, the Inner Product becomes equivalent to Cosine Similarity.

Therefore, `IndexFlatIP` is widely used for semantic search applications.

### Advantages

- Simple implementation.
- Exact similarity search.
- High retrieval accuracy.
- Suitable for small and medium-sized datasets.

## Understanding IndexFlatL2

`IndexFlatL2` performs similarity search using **Euclidean Distance (L2 Distance)**.

Instead of comparing vector directions, it compares the straight-line distance between vectors.

The smaller the distance, the more similar the vectors are.

Unlike `IndexFlatIP`, normalization is not mandatory.

`IndexFlatL2` is useful when Euclidean distance is more appropriate than cosine similarity.

## Difference Between IndexFlatIP and IndexFlatL2

| IndexFlatIP | IndexFlatL2 |
|--------------|-------------|
| Uses Inner Product | Uses Euclidean Distance |
| Works best with normalized vectors | Works with original vectors |
| Equivalent to Cosine Similarity after normalization | Measures actual geometric distance |
| Higher similarity score is better | Smaller distance is better |
| Preferred for semantic search | Preferred for distance-based search |

## Step 14: Add Embeddings to the FAISS Index

After creating the FAISS index, all embedding vectors are inserted into it.

Each embedding is assigned an internal position inside the index.

Later, when a user submits a query, FAISS returns the positions of the most similar embeddings.

### Purpose

Store all embeddings inside the FAISS vector database.

In [16]:
index.add(embeddings)

## Step 15: Create a User Query

Semantic search begins with a user query.

The query can be a sentence, question, or paragraph.

Just like the document chunks, the query must also be converted into an embedding before similarity search can be performed.

### Purpose

Create the user query for semantic search.

In [17]:
query = "Explain Machine Learning?"

## Step 16: Generate the Query Embedding

The query is converted into a dense numerical vector using the same embedding model that was used for the document chunks.

Using the same embedding model ensures that both the document embeddings and the query embedding exist in the same vector space.

The embedding is also converted to **float32**, which is required by FAISS.

### Purpose

Convert the user query into an embedding.

In [18]:
query_embedding = embedding_model.encode(query).astype("float32")

## Step 17: Check the Shape of the Query Embedding

The generated query embedding is initially a one-dimensional array.

However, FAISS expects the input to be a two-dimensional array where each row represents one query vector.

Checking the shape helps verify the current format before reshaping.

### Purpose

Display the shape of the query embedding.

In [19]:
query_embedding.shape

(384,)

## Step 18: Reshape the Query Embedding

The query embedding generated by the Sentence Transformer is a **one-dimensional (1D)** array.

However, FAISS expects the input to be a **two-dimensional (2D)** array where:

- Each row represents one query.
- Each column represents one feature of the embedding.

Since we are searching using only one query, we reshape the embedding into a matrix with **1 row** and **all embedding dimensions**.

### Why Do We Use `reshape(1, -1)`?

The syntax

```python
reshape(1, -1)
```

means:

- `1` → Create one row.
- `-1` → Automatically determine the number of columns.

For example,

Before reshaping

```
(384,)
```

After reshaping

```
(1, 384)
```

Now the embedding is in the format required by FAISS.

### Purpose

Convert the query embedding from a 1D vector into a 2D matrix.

In [20]:
query_embedding = query_embedding.reshape(1, -1)

## Step 19: Normalize the Query Embedding

Just like the document embeddings, the query embedding must also be normalized.

If only the stored embeddings are normalized while the query is not, the similarity scores will not be accurate.

By normalizing both the document embeddings and the query embedding, FAISS performs true cosine similarity search.

### Purpose

Normalize the query embedding before searching the vector database.

In [21]:
faiss.normalize_L2(query_embedding)

# Semantic Similarity Search

The vector database is now ready for searching.

During semantic search, FAISS compares the query embedding with every stored embedding and retrieves the vectors that are most similar.

Unlike keyword search, semantic search retrieves documents based on **meaning** rather than exact word matching.

For example,

Document

```
Machine Learning is a branch of Artificial Intelligence.
```

Query

```
Explain AI and Machine Learning.
```

Even though the wording is different, both sentences have similar meanings.

Semantic search retrieves the document because their embeddings are close in the vector space.

## Step 20: Define the Number of Results (k)

The parameter **k** specifies how many similar documents should be retrieved.

For example,

- `k = 1` → Retrieve only the most similar chunk.
- `k = 3` → Retrieve the three most similar chunks.
- `k = 5` → Retrieve the five most similar chunks.

Increasing **k** provides more context but may also include less relevant documents.

### Purpose

Specify the number of nearest neighbors to retrieve.

In [22]:
k = 3

## Interview Question: What is the Meaning of **k**?

**Question**

What does **k** represent in semantic search?

**Answer**

The parameter **k** represents the number of nearest neighbors (most similar document chunks) that should be retrieved from the vector database.

For example,

- `k = 1` retrieves only the best match.
- `k = 3` retrieves the top three matches.
- `k = 5` retrieves the top five matches.

Choosing the correct value of **k** is important because:

- A very small value may miss useful information.
- A very large value may retrieve irrelevant documents.

Most Retrieval-Augmented Generation (RAG) applications commonly use values between **3 and 10**.

## Step 21: Perform Similarity Search

The `search()` function compares the query embedding with all stored embeddings in the FAISS index.

It returns two outputs:

- **Distances (or Similarity Scores)** – Indicates how similar each retrieved vector is to the query.
- **Indices** – Represents the positions of the retrieved embeddings in the original chunk list.

### Syntax

```python
distances, indices = index.search(query_embedding, k)
```

### Purpose

Retrieve the top-k most similar document chunks.

In [23]:
distances, indices = index.search(query_embedding, k)

## Understanding the Search Output

The `search()` function returns two arrays.

### 1. Distances (Similarity Scores)

The first array contains the similarity score between the query and each retrieved embedding.

For **IndexFlatIP**,

- Higher values indicate greater similarity.
- The highest score corresponds to the best matching document.

Example

```python
[[0.95, 0.91, 0.88]]
```

---

### 2. Indices

The second array contains the positions of the retrieved embeddings.

Example

```python
[[12, 45, 30]]
```

This means

- Chunk 12 is the most similar.
- Chunk 45 is the second most similar.
- Chunk 30 is the third most similar.

These indices are used to retrieve the original text chunks from the `chunks` list.

## Step 22: Display the Similarity Scores

Printing the similarity scores helps us understand how closely each retrieved document matches the user query.

A higher score indicates greater semantic similarity.

### Purpose

Display the similarity scores returned by FAISS.

In [24]:
print(distances)

[[0.619247   0.6190823  0.61763775]]


## Step 23: Display the Retrieved Indices

The retrieved indices indicate the positions of the matching chunks in the original list.

These indices will be used to access and display the actual text.

### Purpose

Display the positions of the retrieved chunks.

In [25]:
print(indices)

[[3189 6726 6574]]


## Step 24: Retrieve the Original Text Chunks

The indices returned by FAISS correspond to positions in the `chunks` list.

Using these indices, we can retrieve and display the original text associated with each matching embedding.

This is the information that will later be passed to a Large Language Model (LLM) in a Retrieval-Augmented Generation (RAG) system.

### Purpose

Display the retrieved text chunks.

In [26]:
for idx in indices[0]:
    print(chunks[idx])
    print("-" * 80)

Machine learning statement 882: A workflow emphasizing data
--------------------------------------------------------------------------------
Machine learning statement 1859: A workflow emphasizing model evaluation
--------------------------------------------------------------------------------
Machine learning statement 1817: A workflow emphasizing gradient
--------------------------------------------------------------------------------


# Complete Pipeline Summary

The vector database pipeline implemented in this notebook follows these steps:

1. Read the dataset.
2. Split the document into overlapping chunks.
3. Generate embeddings for each chunk.
4. Normalize the embeddings.
5. Create a FAISS vector index.
6. Store all embeddings in the index.
7. Accept a user query.
8. Convert the query into an embedding.
9. Normalize the query embedding.
10. Search the FAISS index.
11. Retrieve the top-k most similar document chunks.

These retrieved chunks form the **retrieval component** of a Retrieval-Augmented Generation (RAG) system. The retrieved context can then be provided to a Large Language Model (LLM) to generate accurate, context-aware responses.

# Key Interview Questions

### 1. What is a Vector Database?

A Vector Database stores high-dimensional embeddings and performs semantic similarity search to retrieve documents based on meaning rather than exact keywords.

---

### 2. Why do we generate embeddings?

Embeddings convert text into numerical vectors that capture semantic meaning, enabling similarity search.

---

### 3. Why is chunking required?

Chunking divides large documents into smaller pieces that fit within the embedding model's input limits and improves retrieval accuracy.

---

### 4. Why do we use chunk overlap?

Chunk overlap preserves context between adjacent chunks and reduces information loss at chunk boundaries.

---

### 5. Why is normalization required?

Normalization converts vectors to unit length, allowing cosine similarity to compare vectors fairly without being affected by their magnitude.

---

### 6. What is the difference between IndexFlatIP and IndexFlatL2?

- **IndexFlatIP** uses Inner Product and, after normalization, behaves like Cosine Similarity.
- **IndexFlatL2** uses Euclidean Distance to measure similarity.

---

### 7. What does the parameter `k` represent?

The parameter `k` specifies the number of most similar document chunks to retrieve during similarity search.

---

### 8. What does `index.search()` return?

The `search()` method returns:

- **Distances (Similarity Scores)** – Indicates how similar each retrieved vector is to the query.
- **Indices** – Indicates the positions of the retrieved chunks in the original dataset.

---

### 9. Why must the query use the same embedding model?

Using the same embedding model ensures that both document embeddings and query embeddings exist in the same vector space, making similarity comparisons meaningful.

---

### 10. How does this notebook relate to Retrieval-Augmented Generation (RAG)?

This notebook implements the **retrieval phase** of a RAG system. It retrieves the most relevant document chunks from a vector database, which can then be supplied to a Large Language Model (LLM) to generate context-aware answers.